# US history cache quality audit

This notebook checks the local `data/yfinance_cache/us_history.csv` cache without embedding the 500 MB source file. The intended grain is one row per `code + date`; the check distinguishes structural completeness from source-level price anomalies. Run it from the repository root with a Python environment containing pandas and numpy.

## Scope and assumptions

The cache is a historical daily-bar extract from yfinance, not a real-time feed. A symbol can legitimately have a shorter history because it listed later, was suspended, or is no longer supported by Yahoo; such cases are reported as coverage gaps rather than filled with zeroes. The OHLC rule allows a relative floating-point tolerance of `1e-9` plus an absolute floor of `1e-10`.

In [ ]:
from datetime import date
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
HISTORY = ROOT / 'data' / 'yfinance_cache' / 'us_history.csv'
SPLITS = ROOT / 'data' / 'yfinance_cache' / 'us_splits.csv'
INDUSTRY = ROOT / 'data' / 'cache' / 'us_industry.csv'
REQUIRED = ['date', 'code', 'open', 'high', 'low', 'close', 'volume', 'auto_adjust', 'adjustment_factor']


In [ ]:
rows = 0
codes, dates = set(), set()
null_cells = 0
bad_date = weekend_rows = future_rows = finite_bad = 0
nonpositive_price = negative_volume = strict_ohlc = tolerant_ohlc = 0
previous_key = None
order_violations = adjacent_duplicates = 0
date_counts, code_first, code_last, code_rows = {}, {}, {}, {}

for chunk in pd.read_csv(HISTORY, chunksize=200_000, dtype={'code': 'string'}, low_memory=False):
    rows += len(chunk)
    null_cells += int(chunk[REQUIRED].isna().sum().sum())
    code = chunk['code'].astype('string')
    parsed_date = pd.to_datetime(chunk['date'], errors='coerce')
    date_text = parsed_date.dt.strftime('%Y-%m-%d')
    codes.update(code.dropna())
    dates.update(date_text.dropna())
    bad_date += int(parsed_date.isna().sum())
    weekend_rows += int(parsed_date.dt.dayofweek.isin([5, 6]).fillna(False).sum())
    future_rows += int((parsed_date.dt.date > date.today()).fillna(False).sum())
    numeric = chunk[['open', 'high', 'low', 'close', 'volume', 'adjustment_factor']].apply(pd.to_numeric, errors='coerce')
    finite_bad += int((~np.isfinite(numeric.to_numpy())).sum())
    nonpositive_price += int((numeric[['open', 'high', 'low', 'close']] <= 0).any(axis=1).fillna(False).sum())
    negative_volume += int((numeric['volume'] < 0).fillna(False).sum())
    high_bound = numeric[['open', 'close']].max(axis=1)
    low_bound = numeric[['open', 'close']].min(axis=1)
    strict = (numeric['high'] < high_bound) | (numeric['low'] > low_bound) | (numeric['high'] < numeric['low'])
    scale = numeric[['open', 'high', 'low', 'close']].abs().max(axis=1).fillna(0)
    epsilon = np.maximum(1e-10, scale * 1e-9)
    tolerant = (numeric['high'] + epsilon < high_bound) | (numeric['low'] - epsilon > low_bound) | (numeric['high'] + epsilon < numeric['low'])
    strict_ohlc += int(strict.fillna(False).sum())
    tolerant_ohlc += int(tolerant.fillna(False).sum())
    keys = list(zip(code.tolist(), date_text.tolist()))
    if previous_key is not None and keys and keys[0] < previous_key:
        order_violations += 1
    adjacent_duplicates += sum(keys[i] == keys[i - 1] for i in range(1, len(keys)))
    previous_key = keys[-1] if keys else previous_key
    for symbol, group in chunk.groupby('code', sort=False):
        series = pd.to_datetime(group['date'], errors='coerce')
        code_rows[symbol] = code_rows.get(symbol, 0) + len(group)
        code_first[symbol] = min(code_first.get(symbol, series.min()), series.min())
        code_last[symbol] = max(code_last.get(symbol, series.max()), series.max())
    for day, count in date_text.value_counts().items():
        date_counts[day] = date_counts.get(day, 0) + int(count)

latest = max(dates)
result = {
    'rows': rows, 'unique_codes': len(codes), 'unique_dates': len(dates),
    'date_min': min(dates), 'date_max': max(dates),
    'null_cells': null_cells, 'bad_date': bad_date, 'weekend_rows': weekend_rows,
    'future_rows': future_rows, 'finite_bad_cells': finite_bad,
    'order_violations': order_violations, 'adjacent_duplicate_keys': adjacent_duplicates,
    'nonpositive_price_rows': nonpositive_price, 'negative_volume_rows': negative_volume,
    'strict_ohlc_rows': strict_ohlc, 'tolerant_ohlc_rows': tolerant_ohlc,
    'latest_date': latest, 'latest_date_rows': date_counts[latest],
    'codes_starting_at_min_date': sum(v == pd.Timestamp(min(dates)) for v in code_first.values()),
    'codes_under_500_rows': sum(v < 500 for v in code_rows.values()),
}
print(json.dumps(result, indent=2, default=str))

In [ ]:
splits = pd.read_csv(SPLITS, dtype={'code': 'string'})
industry = pd.read_csv(INDUSTRY, dtype={'code': 'string'})
history_codes = set(codes)
print({
    'split_rows': len(splits),
    'split_codes': int(splits['code'].nunique()),
    'split_duplicate_keys': int(splits.duplicated(['date', 'code']).sum()),
    'split_nonpositive_ratios': int((pd.to_numeric(splits['ratio'], errors='coerce') <= 0).sum()),
    'industry_codes_without_history': len(set(industry['code'].dropna()) - history_codes),
    'history_codes_without_industry': len(history_codes - set(industry['code'].dropna())),
})

## Interpretation

Zero nulls, duplicate keys, malformed dates, future dates, or ordering violations support safe file-level ingestion. Non-positive prices and tolerant OHLC failures remain source-quality exclusions for price-based analysis. For a current-universe completeness claim, compare the history code set with a fresh exchange master because the local industry file is a metadata cache, not an authoritative live security list.